# Hybrid Analytics System Dashboard
MongoDB + Cassandra Analytics

In [11]:
import pandas as pd
import plotly.express as px
from pymongo import MongoClient
from cassandra.cluster import Cluster

In [12]:
mongo = MongoClient('mongodb://localhost:27017')
mongo_db = mongo.hybrid_analytics
products = mongo_db.products

cluster = Cluster(['localhost'])
session = cluster.connect('hybrid_analytics')

print('Connected Successfully')

Connected Successfully


## Products by Category

In [13]:
pipeline = [{'$group': {'_id': '$category', 'count': {'$sum': 1}}}]
category_df = pd.DataFrame(list(products.aggregate(pipeline)))
category_df.columns = ['category','count']
fig = px.bar(
    category_df,
    x='category',
    y='count',
    color='category',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Products by Category'
)
fig.show()

## Products by Brand

In [14]:
pipeline = [{'$group': {'_id': '$brand', 'count': {'$sum': 1}}}]
brand_df = pd.DataFrame(list(products.aggregate(pipeline)))
brand_df.columns = ['brand','count']
fig = px.bar(
    brand_df.sort_values('count'),
    x='count',
    y='brand',
    color='brand',
    orientation='h',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Products by Brand'
)
fig.update_layout(showlegend=False)
fig.show()

## Price Distribution

In [15]:
prices = pd.DataFrame(list(products.find({}, {'price': 1})))
fig = px.histogram(prices, x='price', nbins=30, title='Price Distribution Histogram')
fig.show()


## Average Price by Category

In [16]:
pipeline = [{'$group': {'_id': '$category', 'avg_price': {'$avg': '$price'}}}]
avg_df = pd.DataFrame(list(products.aggregate(pipeline)))
avg_df.columns = ['category','avg_price']

fig = px.bar(
    avg_df,
    x='category',
    y='avg_price',
    color='category',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Average Price by Category'
)
fig.update_layout(showlegend=False)
fig.show()

## Cassandra Metrics

In [17]:
metric_rows = session.execute('SELECT * FROM product_metrics')
metrics_df = pd.DataFrame(list(metric_rows))
metrics_df.head()

,product_id,event_type,count
0,23,view,2
1,114,view,2
2,53,view,5
3,110,view,1
4,91,view,2


In [18]:
event_rows = session.execute('SELECT * FROM product_events')
event_df = pd.DataFrame(list(event_rows))
event_df.head()

,product_id,event_type,event_time
0,23,view,2026-06-23 11:42:11.164
1,23,view,2026-06-23 11:42:10.967
2,114,view,2026-06-23 11:42:10.954
3,114,view,2026-06-23 11:42:10.845
4,53,view,2026-06-23 11:42:12.631


## Top Viewed Products

In [19]:
if 'product_df' not in globals():
    product_df = pd.DataFrame(list(products.find()))
    if 'id' not in product_df.columns and '_id' in product_df.columns:
        try:
            product_df['id'] = product_df['_id'].astype(int)
        except Exception:
            product_df['id'] = product_df['_id'].astype(str)

In [20]:
views_df = metrics_df[metrics_df['event_type'] == 'view'].sort_values('count', ascending=False).head(10)

views_products = views_df.merge(
    product_df[['id', 'title', 'brand']],
    left_on='product_id',
    right_on='id',
    how='left'
)

views_products['label'] = views_products['title'] + ' (' + views_products['brand'] + ')'

fig = px.pie(
    views_products,
    names='label',
    values='count',
    color='label',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Top Viewed Products'
)
fig.show()

## Top Favorited Products

In [21]:
fav_df = metrics_df[metrics_df['event_type'] == 'favorite'].sort_values('count', ascending=False).head(10)
fav_products = fav_df.merge(
    product_df[['id', 'title', 'brand']],
    left_on='product_id',
    right_on='id',
    how='left'
)
fav_products['label'] = fav_products['title'] + ' (' + fav_products['brand'] + ')'

fig = px.pie(
    fav_products,
    names='label',
    values='count',
    color='label',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Top Favorited Products'
)
fig.show()

## Top Purchased Products

In [22]:
buy_df = metrics_df[metrics_df['event_type'] == 'buy'].sort_values('count', ascending=False).head(10)
buy_products = buy_df.merge(
    product_df[['id', 'title', 'brand']],
    left_on='product_id',
    right_on='id',
    how='left'
)
buy_products['label'] = buy_products['title'] + ' (' + buy_products['brand'] + ')'

fig = px.pie(
    buy_products,
    names='label',
    values='count',
    color='label',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Top Purchased Products'
)
fig.show()

## Product Engagement Score

In [23]:
fig = px.bar(
    brand_df,
    x='brand',
    y='count',
    color='brand',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Products by Brand'
)
fig.update_layout(showlegend=False)
fig.show()

In [24]:
metrics_pivot = (
    metrics_df
    .pivot(index='product_id', columns='event_type', values='count')
    .reset_index()
    .fillna(0)
)
metrics_pivot.columns.name = None
metrics_pivot = metrics_pivot.astype({'buy': int, 'favorite': int, 'view': int})

metrics_pivot = metrics_pivot.merge(
    product_df[['id', 'title', 'brand']],
    left_on='product_id',
    right_on='id',
    how='left'
)
metrics_pivot.head()

,product_id,buy,favorite,view,id,title,brand
0,3,0,0,6,3,Polarized motivating core,Samsung
1,5,0,0,3,5,Visionary radical Graphical User Interface,Sony
2,6,1,0,2,6,Robust multimedia open system,Puma
3,7,0,0,3,7,Pre-emptive non-volatile functionalities,Adidas
4,8,0,0,3,8,Visionary national project,Apple


In [25]:
metric_rows = session.execute('SELECT * FROM product_metrics')
metrics_df = pd.DataFrame(list(metric_rows))
metrics_df = metrics_df.sort_values('product_id')
metrics_df.head(10)

,product_id,event_type,count
254,3,view,6
18,5,view,3
161,6,buy,1
162,6,view,2
142,7,view,3
72,8,view,3
179,9,favorite,1
180,9,view,2
33,10,view,3
62,11,view,3


In [27]:

event_rows = session.execute('SELECT * FROM product_events')
event_df = pd.DataFrame(list(event_rows))
event_df = event_df.sort_values('event_time', ascending=False)
print("Total Events:", len(event_df))
event_df.head()

Total Events: 500


,product_id,event_type,event_time
246,149,view,2026-06-23 11:42:12.707
131,151,buy,2026-06-23 11:42:12.702
34,28,view,2026-06-23 11:42:12.697
363,180,view,2026-06-23 11:42:12.692
364,180,view,2026-06-23 11:42:12.687
